# Imports

In [21]:
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.express as px

import numpy as np
import pandas as pd
import seaborn as sns
from ppmi_utils import (
    cohort_map,
    event_id_to_visit,
    load_ppmi_csvs,
    merge_ppmi_tables,
    parse_imaging_protocol,
    plot_bar,
    plot_hist,
    primdiag_map,
    visit_to_event_id,
)

CURRENT_DIR = Path(os.getcwd()).resolve()
ROOT_DIR = CURRENT_DIR.parents[1]
PPMI_CLINICAL = ROOT_DIR / "csv_dir" / "PPMI_CSV" / "CLINICAL"
PPMI_OTHERS = ROOT_DIR / "csv_dir" / "PPMI_CSV" / "OTHERS"
curated_data = PPMI_OTHERS / "PPMI_Curated_Data_Cut_Public_20251112.xlsx"
# https://github.com/neurodatascience/nipoppy-ppmi/blob/main/imaging_descriptions/ppmi_imaging_descriptions.json
ppmi_imaging_descriptions_path = CURRENT_DIR / "ppmi_imaging_descriptions.json"
ppmi_imaging_ignored_path = CURRENT_DIR / "ppmi_imaging_ignored.csv"

# CSV preprocessing

### Uncurated clinical tabular data 

Not cureated, so might not be used unless missing info in the curated data need to be found here

In [22]:
ppmi_data = load_ppmi_csvs(PPMI_CLINICAL)
print(f"Available dataframes: {list(ppmi_data.keys())}")

Available dataframes: ['Clock_Drawing', 'MDS_UPDRS_Part_II__Patient_Questionnaire', 'Primary_Clinical_Diagnosis', 'University_of_Pennsylvania_Smell_Identification_Test_UPSIT', 'Montreal_Cognitive_Assessment__MoCA_', 'MDS-UPDRS_Part_IV__Motor_Complications', 'MDS-UPDRS_Part_I_Patient_Questionnaire', 'Modified_Boston_Naming_Test', 'Demographics', 'Benton_Judgement_of_Line_Orientation', 'Symbol_Digit_Modalities_Test', 'Modified_Semantic_Fluency', 'Age_at_visit', 'Letter_-_Number_Sequencing', 'MDS-UPDRS_Part_III', 'Participant_Status', 'MDS-UPDRS_Part_I', 'Clinical_Diagnosis', 'Socio-Economics']


In [23]:
uncurated_clinical_df = merge_ppmi_tables(ppmi_data)
print("Uncurated DataFrame shape:", uncurated_clinical_df.shape)
print("Uncurated DataFrame columns:", uncurated_clinical_df.columns.tolist())

Uncurated DataFrame shape: (54734, 461)
Uncurated DataFrame columns: ['REC_ID', 'PATNO', 'EVENT_ID', 'PAG_NAME', 'INFODT', 'CLCKPII', 'CLCK2HND', 'CLCKNMRK', 'CLCKNUIN', 'CLCKALNU', 'CLCKNUSP', 'CLCKNUED', 'CLCKTOT', 'AGE_ASSESS_CLCKDRAW', 'DVT_CLCKDRAW', 'DVZ_CLCKDRAW', 'ORIG_ENTRY', 'LAST_UPDATE', 'REC_ID_MDS_UPDRS_Part_II__Patient_Questionnaire', 'PAG_NAME_MDS_UPDRS_Part_II__Patient_Questionnaire', 'INFODT_MDS_UPDRS_Part_II__Patient_Questionnaire', 'NUPSOURC', 'NP2SPCH', 'NP2SALV', 'NP2SWAL', 'NP2EAT', 'NP2DRES', 'NP2HYGN', 'NP2HWRT', 'NP2HOBB', 'NP2TURN', 'NP2TRMR', 'NP2RISE', 'NP2WALK', 'NP2FREZ', 'NP2PTOT', 'ORIG_ENTRY_MDS_UPDRS_Part_II__Patient_Questionnaire', 'LAST_UPDATE_MDS_UPDRS_Part_II__Patient_Questionnaire', 'REC_ID_Primary_Clinical_Diagnosis', 'PAG_NAME_Primary_Clinical_Diagnosis', 'INFODT_Primary_Clinical_Diagnosis', 'PRIMDIAG', 'NEWDIAGEXP', 'OTHNEURO', 'DXLVL', 'ORIG_ENTRY_Primary_Clinical_Diagnosis', 'LAST_UPDATE_Primary_Clinical_Diagnosis', 'REC_ID_University_of_Pen

### Official curated merge tabular data

In [24]:
curated_clinical_df = pd.read_excel(curated_data)
print("Curated Clinical DataFrame shape:", curated_clinical_df.shape)
print("Curated Clinical DataFrame columns:", curated_clinical_df.columns.tolist())

Curated Clinical DataFrame shape: (17252, 179)
Curated Clinical DataFrame columns: ['SITE', 'PATNO', 'COHORT', 'subgroup', 'enroll_phase', 'enroll_source', 'analytic_subgroup', 'HIQ_RBD', 'study_status', 'NSD_Status', 'NSD_STAGE', 'PRIMDIAG', 'OTHNEURO', 'EVENT_ID', 'YEAR', 'visit_date', 'age', 'age_at_visit', 'SEX', 'EDUCYRS', 'race', 'HISPLAT', 'ASHKJEW', 'AFICBERB', 'BASQUE', 'fampd', 'fampd_bin', 'handed', 'howlive', 'sex_orient', 'BMI', 'agediag', 'ageonset', 'duration', 'duration_yrs', 'DOMSIDE', 'sym_tremor', 'sym_rigid', 'sym_brady', 'sym_posins', 'sym_other', 'sym_unknown', 'PDTRTMNT', 'LEDD', 'age_datscan', 'age_LP', 'age_upsit', 'upsit', 'upsit_pctl', 'upsit_pctl15', 'moca', 'bjlot', 'DVS_JLO_MSSA', 'DVS_JLO_MSSAE', 'clockdraw', 'DVT_CLCKDRAW', 'DVZ_CLCKDRAW', 'hvlt_discrimination', 'hvlt_immediaterecall', 'hvlt_retention', 'HVLTFPRL', 'HVLTRDLY', 'HVLTREC', 'DVT_TOTAL_RECALL', 'DVT_DELAYED_RECALL', 'DVT_RETENTION', 'DVT_RECOG_DISC_INDEX', 'lexical', 'DVT_FAS', 'DVS_FAS', 'l

### idaSearch (image info)

In [25]:
# ida search csv load and preprocessing
ida_df = pd.read_csv(PPMI_OTHERS / "idaSearch_18Feb2026.csv", low_memory=False)
ida_df = ida_df[~ida_df["Modality"].isin(["Path", "CT"])].copy()
ida_df["Weight"] = ida_df["Weight"].replace(0, np.nan)
ida_df["Age"] = ida_df["Age"].replace(0, np.nan)
ida_df["Study Date"] = pd.to_datetime(ida_df["Study Date"], errors="coerce")
ida_df = ida_df.rename(columns={"Subject ID": "PATNO"})

# only original, no preprocessed or unknown
ida_df = ida_df[(ida_df["Type"] == "Original") | (ida_df["Type"].isna())].copy()

# Apply parsing
protocol_parsed = ida_df["Imaging Protocol"].apply(parse_imaging_protocol)

# Automatically get all unique keys
all_keys = set()
protocol_parsed.apply(lambda d: all_keys.update(d.keys()))

# Create new columns dynamically
for key in all_keys:
    ida_df[key] = protocol_parsed.apply(lambda x: x.get(key, np.nan))


print("ida searchDataFrame shape:", ida_df.shape)
print(ida_df.columns.tolist())


ida searchDataFrame shape: (57651, 34)
['PATNO', 'Project', 'Sex', 'Weight', 'Research Group', 'Visit', 'Study Date', 'Archive Date', 'Age', 'GDSCALE Total Score', 'Modality', 'Description', 'Type', 'Imaging Protocol', 'Image ID', 'Structure', 'Laterality', 'Image Type', 'Registration', 'Tissue', 'Mfg Model', 'Acquisition Type', 'Acquisition Plane', 'Radioisotope', 'Frames', 'Manufacturer', 'TE', 'Matrix Z', 'Slice Thickness', 'Field Strength', 'Weighting', 'TR', 'Gradient Directions', 'Radiopharmaceutical']


# Advanced modalities 

### setup

In [26]:
# avanced modalities, normalize by stripping and uppercasing
with open(ppmi_imaging_descriptions_path, "r") as f:
    modality_dict = json.load(f)


def normalize_list(lst):
    return [s.strip().upper() for s in lst]


dwi_set = set(normalize_list(modality_dict["dwi"]))
func_set = set(normalize_list(modality_dict["func"]))
anat_sets = {k: set(normalize_list(v)) for k, v in modality_dict["anat"].items()}

# filter out ignored descriptions
ignore_df = pd.read_csv(ppmi_imaging_ignored_path)
ignore_set = set(ignore_df["Description"].str.strip().str.upper())
ida_df["Description_clean"] = ida_df["Description"].astype(str).str.strip().str.upper()
ida_df = ida_df[~ida_df["Description_clean"].isin(ignore_set)].copy()

In [27]:
def classify_advanced(row):
    desc = row["Description_clean"]
    modality = row["Modality"]
    # --- DWI ---
    if desc in dwi_set:
        return "DWI"
    # --- Functional ---
    if desc in func_set:
        return "rsfMRI"
    # --- Anatomical ---
    for anat_type, anat_set in anat_sets.items():
        if desc in anat_set:
            return anat_type  # T1w, T2w, T2starw, FLAIR
    # --- Nuclear imaging ---
    if modality == "PET":
        return "PET"
    if modality == "SPECT":
        return "SPECT"
    return "Unknown"


ida_df["Advanced_Modality"] = ida_df.apply(classify_advanced, axis=1)

print(ida_df["Advanced_Modality"].value_counts())
print(pd.crosstab(ida_df["Modality"], ida_df["Advanced_Modality"]))
# the "Modality" column cannot be trusted, some DTI are in MRI and vice versa,
# nor the "Weighting" (under Imaging Protocol)
# so rely on the "Advanced_Modality" column, which is derived from the descriptions, for modality-specific analyses

# print the number of adavancec modality in unknwn
# there should be none, otherwise the description json is incomplete and needs to be updated
print("Number of Unknown modality:", (ida_df["Advanced_Modality"] == "Unknown").sum())
print("Liste of descriptions classified as Unknown:")
print(ida_df[ida_df["Advanced_Modality"] == "Unknown"]["Description_clean"].unique())
if (ida_df["Advanced_Modality"] == "Unknown").sum() > 0:
    print(
        "Warning: There are descriptions classified as Unknown. Please check the list above and update the imaging descriptions JSON if needed."
    )

Advanced_Modality
T1w        17574
DWI        15922
rsfMRI      7503
T2w         6115
SPECT       4551
FLAIR       4290
PET          762
T2starw        5
Name: count, dtype: int64
Advanced_Modality    DWI  FLAIR  PET  SPECT    T1w  T2starw   T2w  rsfMRI
Modality                                                                 
DTI                13120      0    0      0     20        0    36       0
MRI                 2714   4290    0      0  17263        5  6079    3631
PET                    0      0  762      0      0        0     0       0
SPECT                 20      0    0   4551      0        0     0       0
fMRI                  68      0    0      0    291        0     0    3872
Number of Unknown modality: 0
Liste of descriptions classified as Unknown:
<StringArray>
[]
Length: 0, dtype: str


### Merge clinical and imaging info

In [28]:
# merge clinical and imaging dataframes
ida_df["PATNO"] = ida_df["PATNO"].astype(str)
curated_clinical_df["PATNO"] = curated_clinical_df["PATNO"].astype(str)
ida_df["EVENT_ID"] = ida_df["Visit"].map(visit_to_event_id)
curated_clinical_df["PRIMDIAG_DESC"] = curated_clinical_df["PRIMDIAG"].map(primdiag_map)
curated_clinical_df["COHORT_DESC"] = curated_clinical_df["COHORT"].map(cohort_map)

df = pd.merge(
    curated_clinical_df,
    ida_df,
    # how="left",  # left merge to keep all clinical data, even those without imaging, but results in many NaNs in imaging columns
    # how="right",  # right merge to keep all imaging data, but loses clinical data for subjects without imaging
    how="outer",  # to keep all data, but results in many NaNs in both clinical and imaging columns
    on=["PATNO", "EVENT_ID"],
    suffixes=("", "_idasearch"),  # Curated stays same, Imaging gets '_ida' suffix
)
df["Visit"] = df["EVENT_ID"].map(event_id_to_visit)

print("Final merged DataFrame shape:", df.shape)
print("Final merged DataFrame columns:", df.columns.tolist())
print(df["EVENT_ID"].unique().tolist())

Final merged DataFrame shape: (67833, 216)
Final merged DataFrame columns: ['SITE', 'PATNO', 'COHORT', 'subgroup', 'enroll_phase', 'enroll_source', 'analytic_subgroup', 'HIQ_RBD', 'study_status', 'NSD_Status', 'NSD_STAGE', 'PRIMDIAG', 'OTHNEURO', 'EVENT_ID', 'YEAR', 'visit_date', 'age', 'age_at_visit', 'SEX', 'EDUCYRS', 'race', 'HISPLAT', 'ASHKJEW', 'AFICBERB', 'BASQUE', 'fampd', 'fampd_bin', 'handed', 'howlive', 'sex_orient', 'BMI', 'agediag', 'ageonset', 'duration', 'duration_yrs', 'DOMSIDE', 'sym_tremor', 'sym_rigid', 'sym_brady', 'sym_posins', 'sym_other', 'sym_unknown', 'PDTRTMNT', 'LEDD', 'age_datscan', 'age_LP', 'age_upsit', 'upsit', 'upsit_pctl', 'upsit_pctl15', 'moca', 'bjlot', 'DVS_JLO_MSSA', 'DVS_JLO_MSSAE', 'clockdraw', 'DVT_CLCKDRAW', 'DVZ_CLCKDRAW', 'hvlt_discrimination', 'hvlt_immediaterecall', 'hvlt_retention', 'HVLTFPRL', 'HVLTRDLY', 'HVLTREC', 'DVT_TOTAL_RECALL', 'DVT_DELAYED_RECALL', 'DVT_RETENTION', 'DVT_RECOG_DISC_INDEX', 'lexical', 'DVT_FAS', 'DVS_FAS', 'lns', 'DV

/tmp/ipykernel_3568788/69788571.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  curated_clinical_df["PRIMDIAG_DESC"] = curated_clinical_df["PRIMDIAG"].map(primdiag_map)
/tmp/ipykernel_3568788/69788571.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  curated_clinical_df["COHORT_DESC"] = curated_clinical_df["COHORT"].map(cohort_map)


# Filters

General filters

In [29]:
print("Before filter:")
print("Shape ", df.shape)
print("unique patient ", df["PATNO"].nunique())
print("unique patient-visit pairs ", df.groupby(["PATNO", "EVENT_ID"]).ngroups)

Before filter:
Shape  (67833, 216)
unique patient  5695
unique patient-visit pairs  20521


In [30]:
# only original, no preprocessed or unknown
df = df[(df["Type"] == "Original") | (df["Type"].isna())].copy()

# the protocol 2.0 started in august 2020, so we keep only data from then on to have a more homogeneous dataset
# and also because older data is more likely to be of lower quality
df = df[df["Study Date"] >= pd.Timestamp("2020-09-01")].copy()

# filter on descriptions, remove the raws with description that contains one of ['phantom', 'adc', 'trace', localizer']
df = df[
    ~df["Description_clean"].str.contains("PHANTOM|ADC|TRACE|LOCALIZER", na=False)
].copy()

# Most relevant modalities. Neuromanalin is included in T1w  (BIDS convention ?)
df = df[(df["Advanced_Modality"].isin(["DWI", "T1w"]))].copy()

# from august 2020 (PPMI 2.0), the research groups are the following 3: Control, PD, Prodromal.
# so we discrdard other groups (SWEDD, Genetic Cohort, Genetic Registry) which are smaller and more heterogeneous
df = df[(df["Research Group"].isin(["Control", "PD", "Prodromal"]))].copy()

# at least 32 gradient directions for DWI
df = df[(df["Advanced_Modality"] != "DWI") | (df["Gradient Directions"] >= 32)].copy()


# Keep only rows where a description is used by at least n different patients
occurence_threshold = 5
df = df[
    df.groupby("Description_clean")["PATNO"].transform("nunique") >= occurence_threshold
].copy()


# df = df[
#     ~(df["Advanced_Modality"].isin(["rsfMRI"]))
#     # ~(df["Advanced_Modality"].isin(["rsfMRI","MRI_other", "DTI_other", "fMRI_other", "Unknown"]))
# ].copy()

# df = df[
#     (
#         (df["Advanced_Modality"] != "T1w")
#         | ((df["Acquisition Type"] == "3D") | (df["Matrix Z"] >= 70))
#     )
# ].copy()


In [31]:
print("After filter:")
print("Shape ", df.shape)
print("unique patient ", df["PATNO"].nunique())
print("unique patient-visit pairs ", df.groupby(["PATNO", "EVENT_ID"]).ngroups)

After filter:
Shape  (22744, 216)
unique patient  2126
unique patient-visit pairs  3213


In [32]:
# all the columns in the final dataframe
print(df.columns.tolist())

['SITE', 'PATNO', 'COHORT', 'subgroup', 'enroll_phase', 'enroll_source', 'analytic_subgroup', 'HIQ_RBD', 'study_status', 'NSD_Status', 'NSD_STAGE', 'PRIMDIAG', 'OTHNEURO', 'EVENT_ID', 'YEAR', 'visit_date', 'age', 'age_at_visit', 'SEX', 'EDUCYRS', 'race', 'HISPLAT', 'ASHKJEW', 'AFICBERB', 'BASQUE', 'fampd', 'fampd_bin', 'handed', 'howlive', 'sex_orient', 'BMI', 'agediag', 'ageonset', 'duration', 'duration_yrs', 'DOMSIDE', 'sym_tremor', 'sym_rigid', 'sym_brady', 'sym_posins', 'sym_other', 'sym_unknown', 'PDTRTMNT', 'LEDD', 'age_datscan', 'age_LP', 'age_upsit', 'upsit', 'upsit_pctl', 'upsit_pctl15', 'moca', 'bjlot', 'DVS_JLO_MSSA', 'DVS_JLO_MSSAE', 'clockdraw', 'DVT_CLCKDRAW', 'DVZ_CLCKDRAW', 'hvlt_discrimination', 'hvlt_immediaterecall', 'hvlt_retention', 'HVLTFPRL', 'HVLTRDLY', 'HVLTREC', 'DVT_TOTAL_RECALL', 'DVT_DELAYED_RECALL', 'DVT_RETENTION', 'DVT_RECOG_DISC_INDEX', 'lexical', 'DVT_FAS', 'DVS_FAS', 'lns', 'DVS_LNS', 'MODBNT', 'DVS_BNT', 'PCTL_BNT', 'SDMTOTAL', 'DVT_SDM', 'DVSD_SDM',

In [33]:
print(df["EVENT_ID"].unique().tolist())

['BL', 'V06', 'V10', 'V04', 'V02', 'V12']


# Descriptions

In [34]:
description_clean_list = df["Description_clean"].unique().tolist()
print("Number of unique cleaned descriptions:", len(description_clean_list))
print(description_clean_list)
print(df["Advanced_Modality"].unique().tolist())

# Get the list
desc_list = df["Description_clean"].unique().tolist()

# Save to a txt file
with open("descriptions.txt", "w", encoding="utf-8") as f:
    for item in desc_list:
        f.write(str(item) + "\n")

# print descrptions in order of frequency
desc_freq = df["Description_clean"].value_counts()
print(desc_freq)

print(df["Description_clean"].unique().tolist())

Number of unique cleaned descriptions: 72
['SAG 3D MPRAGE', 'SAG 3D T1 FSPGR', 'AXIAL 2D GRE-MT', 'AX DTI', '3D T1 _WEIGHTED', '2D GRE-MT', 'DTI_REVB0_AP', 'DTI_B0_PA', 'DTI_B2000_64DIR_PA', 'DTI_B700_64DIR_PA', 'DTI_B1000_64DIR_PA', '3D T1-WEIGHTED', '2D GRE-NM_MT', 'SAG FSPGR 3D VOLUMETRIC T1', '3D_T1-WEIGHTED', '2D_GRE-MT', 'MPRAGE - SAG', '2D GRE-MT_ACPC', 'AXIAL DTI FREQ A_P', 'AX GRE -MT', 'SAG 3D T1', '3D-T1-WEIGHTED_SAGITAL', '2D GRE-NM', '3D T1', 'NM - MT', 'MPRAGE SAG IPAT ISO', 'NM-GRE', 'T1-WEIGHTED, 3D VOLUMETRIC', 'DTI_RL', 'DTI_LR', '3D T1-WEIGHTED MPRAGE', '3D T1 MPRAGE', '2D GRE MT', 'SAG 3D T1-WEIGHTED', '3D T1-WEIGHTED_ND', 'SAG 3D MPR', '3D SAG T1 MPRAGE', 'SAG 3D T1 MPRAGE', '2D GRE-MT Q9R1007332', 'SAG 3D T1 MPRAGE Q9R1007332', 'T1 3D VOLUMETRIC SEQUENCE', 'VT1 3D VOLUMETRIC SEQUENCE', 'NM-MT', '3D  T1W', '2D GRE-MT OBL PAR TO AC PC LINES COVER THALA TO BOT OF 3 VENTICL', '2D GRE-MT 1', 'AX DTI _LR', 'AX DTI _RL', '2D GRE-MT 2', '2D GRE-MT 3', '2D GRE-MT 4', '2D G

In [35]:
desc_counts = df["Description_clean"].value_counts()
top_df = desc_counts.reset_index()
top_df.columns = ["Description", "Count"]

# Reverse order for horizontal readability (small → big from top to bottom)
top_df = top_df.iloc[::-1]

fig = px.bar(
    top_df,
    x="Count",
    y="Description",
    orientation="h",
    title="Descriptions",
    height=2000,
)

# Remove categoryorder — let your dataframe control it
fig.update_layout(
    yaxis={"categoryorder": "array", "categoryarray": top_df["Description"]}
)
fig.update_yaxes(
    tickmode="array",
    tickvals=top_df["Description"],
    ticktext=top_df["Description"],
    tickfont=dict(size=8),  # shrink font so all labels fit
)
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

# Filter VISIT or PATIENTS based on required image modalities

possible values are

DWI, rsfMRI, T1w, T2w, T2starw, FLAIR, PET, SPECT

MRI_other
fMRI_other
DTI_other
Unknown


In [ ]:
# # Choose filtering mode
# strict_per_visit = False  # True = must have all modalities in a single visit, False = across all visits

# # Define required modalities
# required_modalities = {"T1w", "DWI"}

# if strict_per_visit:
#     # ----------------------------
#     # Strict: visit must have all modalities
#     # ----------------------------
#     visit_modalities = (
#         df.groupby(["PATNO", "EVENT_ID"])["Advanced_Modality"].unique().reset_index()
#     )
#     visit_modalities["has_all"] = visit_modalities["Advanced_Modality"].apply(
#         lambda mods: required_modalities.issubset(set(mods))
#     )
#     valid_visits = visit_modalities.loc[
#         visit_modalities["has_all"], ["PATNO", "EVENT_ID"]
#     ]
#     df_filtered = df.merge(valid_visits, on=["PATNO", "EVENT_ID"], how="inner")

# else:
#     # ----------------------------
#     # Less strict: patient must have all modalities across all visits
#     # ----------------------------
#     patient_modalities = (
#         df.groupby("PATNO")["Advanced_Modality"]
#         .apply(lambda mods: set(mods))
#         .reset_index()
#     )
#     patient_modalities["has_all"] = patient_modalities["Advanced_Modality"].apply(
#         lambda mods: required_modalities.issubset(mods)
#     )
#     valid_patients = patient_modalities.loc[patient_modalities["has_all"], "PATNO"]
#     df_filtered = df[df["PATNO"].isin(valid_patients)].copy()


# print(f"Original rows: {len(df)}, filtered rows: {len(df_filtered)}")
# print("unique patients:", df_filtered["PATNO"].nunique())
# print("unique patient-visit pairs:", df_filtered.groupby(["PATNO", "EVENT_ID"]).ngroups)

# # # Update df
# df = df_filtered.copy()


In [ ]:
# print percetage of na values in each column in the following list
col_list = ["PRIMDIAG", "COHORT", "Research Group", "subgroup", "Image ID"]
for col in col_list:
    na_percentage = df[col].isna().mean() * 100
    print(f"{col}: {na_percentage:.2f}% NA values")

# Basic analysis

### Patients and visits without images

In [ ]:
# Visits without image ID
df_no_image_id = df[df["Image ID"].isna()]
n_visits_no_image = df_no_image_id.groupby(["PATNO", "EVENT_ID"]).ngroups
print(f"Number of visits without 'Image ID': {n_visits_no_image}")

total_visits = df.groupby(["PATNO", "EVENT_ID"]).ngroups
print(f"Total number of visits: {total_visits}")
print(
    f"Percentage of visits without 'Image ID': {n_visits_no_image / total_visits:.2%}"
)

In [ ]:
# Number of unique patients with at least one visit without image ID
# Total number of patients
total_patients = df["PATNO"].nunique()
# Number of patients with no images at all
patients_with_image = df.loc[df["Image ID"].notna(), "PATNO"].unique()
patients_no_image = set(df["PATNO"]) - set(patients_with_image)
n_patients_no_image = len(patients_no_image)

print(f"Total patients: {total_patients}")
print(f"Patients with no image at all: {n_patients_no_image}")
print(f"Percentage with no image: {n_patients_no_image / total_patients:.2%}")


### Age at baseline

In [ ]:
baseline_df = df[df["EVENT_ID"] == "BL"].copy()
baseline_unique = baseline_df.drop_duplicates(subset=["PATNO"])  # keep first occurence
print("Baseline age summary (one row per patient):")
print(baseline_unique[["age"]].describe().T)
plot_hist(baseline_unique, "age", title="Age at baseline")

### Sex distribution

In [ ]:
# Count occurrences
# Get most frequent Sex per patient
sex_per_patient = df.groupby("PATNO")["Sex"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)

# Count distribution
sex_counts = sex_per_patient.value_counts(dropna=False)

print("Sex distribution per patient:")
print(sex_counts)

# Print table
print("Sex distribution at baseline:")
print(sex_counts.to_frame().T)

# Plot pie chart
plt.figure(figsize=(6, 6))
plt.pie(
    sex_counts,
    labels=sex_counts.index,
    autopct="%1.1f%%",  # show percentage
    startangle=90,  # rotate to start from top
)
plt.title("Sex distribution")
plt.axis("equal")  # equal aspect ratio ensures pie is circular
plt.show()


### Cohort definition
1. Parkinson’s Disease, i.e., people who have a formal diagnosis of Parkinson’s disease (PD) 
2. Prodromal, i.e., people who are at risk of developing PD based on clinical features, genetic variants or other biomarkers but have not been formally diagnosed 
3. Healthy Controls, i.e., people with no neurologic disorder and no first-degree relative with PD
4. SWEDD (Scan without dopaminergic deficit). This is a small legacy cohort that you may wish to exclude, depending on your research purpose; for more details, see https://www.ppmi-info.org/study-design/study-cohorts#legacy/

In [ ]:
# diagnosis at baseline
print("Diagnosis distribution at baseline:")
cohort_counts = baseline_unique["Research Group"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "Research Group",
    title="Research Group distribution at baseline",
)

In [ ]:
cohort_counts = baseline_unique["COHORT_DESC"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "COHORT_DESC",
    title="Cohort distribution at baseline",
)

### subgroup
Subgroup is derived from various source columns to give a more detailed group assignment than cohort. It can take values of Healthy Control, SWEDD, SWEDD/PD, SWEDD/nonPD, Hyposmia, RBD, Sporadic PD, LRRK2, GBA, PINK1, PRKN, SNCA or combinations of genetic variants and/or RBD (e.g. LRRK2 + GBA, GBA + RBD).

In [ ]:
cohort_counts = baseline_unique["subgroup"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "subgroup",
    title="Subgroups distribution at baseline",
)

### PRIMDIAG

In [ ]:
cohort_counts = baseline_unique["PRIMDIAG_DESC"].value_counts(dropna=False)
# print(cohort_counts.to_frame().T)
plot_bar(
    baseline_unique.reset_index(drop=True),
    "PRIMDIAG_DESC",
    title="PRIMDIAG distribution at baseline",
)

### cross tab for diag, subgroup, research group

In [ ]:
print(pd.crosstab(baseline_unique["PRIMDIAG_DESC"], baseline_unique["subgroup"]))


# Create crosstab
ct = pd.crosstab(
    baseline_unique["PRIMDIAG_DESC"],
    # baseline_unique["Research Group"],
    baseline_unique["subgroup"],
)

# Optional: sort by total frequency
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]

# Plot heatmap
plt.figure(figsize=(10, 12))

sns.heatmap(
    ct,
    annot=True,  # show raw numbers
    fmt="d",  # integer formatting
    cmap="Blues",
    linewidths=0.5,
    cbar_kws={"label": "Count"},
)

plt.title("Primary Diagnosis by Subgroup (Raw Counts)")
plt.xlabel("Subgroup")
plt.ylabel("Primary Diagnosis")
plt.tight_layout()
plt.show()

In [ ]:
# series1 = baseline_unique["Research Group"]
# series2 = baseline_unique["PRIMDIAG_DESC"]
# # series2 = baseline_unique["subgroup"]
# print(pd.crosstab(series2, series1))

# Longitudinal analysis

### Cumulative number of visits through years

In [ ]:
import plotly.express as px
import pandas as pd

# Ensure Study Date is datetime
df["Study Date"] = pd.to_datetime(df["Study Date"], errors="coerce")

# Keep only unique patient-event pairs
unique_visits = df.drop_duplicates(subset=["PATNO", "EVENT_ID"])
unique_visits = unique_visits[unique_visits["Study Date"] >= "2010-01-01"]

# Extract Year-Month
unique_visits["YearMonth"] = unique_visits["Study Date"].dt.to_period("M")

# Count visits per month
visits_per_month = unique_visits.groupby("YearMonth").size()
visits_per_month = visits_per_month.cumsum().reset_index()
visits_per_month.columns = ["YearMonth", "CumulativeVisits"]

# Convert YearMonth to timestamp for Plotly
visits_per_month["YearMonth"] = visits_per_month["YearMonth"].dt.to_timestamp()

# Plot interactive line chart
fig = px.line(
    visits_per_month,
    x="YearMonth",
    y="CumulativeVisits",
    title="Cumulative Number of Unique Visits per Month",
    markers=True,
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Unique Visits",
    xaxis=dict(rangeslider=dict(visible=True)),  # adds a zoomable range slider
    hovermode="x unified",
)

fig.show()

### How many visits per patient ?

In [ ]:
visits_per_patient = df.groupby("PATNO")["EVENT_ID"].nunique()
print("Visits per patient (summary):")
print(visits_per_patient.describe().to_frame().T)
# Histogram of visit counts
plot_hist(
    visits_per_patient.reset_index(), "EVENT_ID", title="Number of visits per patient"
)

### Longitudinal diagnosis changes

There is no change for 
column_name_for_transition = "COHORT_DESC"
column_name_for_transition = "Research Group"

In [ ]:
column_name_for_transition = "PRIMDIAG_DESC"

# number of distinct diagnoses per patient
diagnosis_over_time = df.groupby("PATNO")[column_name_for_transition].nunique()

# Patients with >1 diagnosis
patients_changed_diag = diagnosis_over_time[diagnosis_over_time > 1]
print(f"Number of patients with diagnosis change: {len(patients_changed_diag)}")

# diagnosis_per_patient = df.groupby("PATNO")[column_name].unique()
# for pat in patients_changed_diag.index:
#     print(f"{pat}: {diagnosis_per_patient[pat]}")

# Plot distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=diagnosis_over_time, color="skyblue")
plt.title("Number of distinct diagnoses per patient")
plt.xlabel("Distinct diagnoses count")
plt.ylabel("Number of patients")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.show()

In [ ]:
# Ensure visits are ordered in time
df_sorted = df.sort_values(["PATNO", "Study Date"])

transitions = []

for pat, group in df_sorted.groupby("PATNO"):
    diagnoses = group[column_name_for_transition].dropna().unique()

    if len(diagnoses) > 1:
        # Get ordered diagnoses per visit (not just unique)
        ordered_diag = (
            group[[column_name_for_transition, "Study Date"]]
            .dropna()
            .sort_values("Study Date")[column_name_for_transition]
            .tolist()
        )

        # Remove consecutive duplicates
        cleaned = [ordered_diag[0]]
        for d in ordered_diag[1:]:
            if d != cleaned[-1]:
                cleaned.append(d)

        # Create pairwise transitions
        for i in range(len(cleaned) - 1):
            transitions.append(f"{cleaned[i]} → {cleaned[i + 1]}")


transition_counts = pd.Series(transitions).value_counts()
print(len(transitions))
transition_counts = transition_counts.head(5)  # keep top 10 transitions

# Total number of transitions
total_transitions = transition_counts.sum()

plt.figure(figsize=(10, 5))

ax = sns.barplot(
    x=transition_counts.values, y=transition_counts.index, color="steelblue"
)

plt.title("Diagnosis Transitions (Patients with Change)")
plt.xlabel("Number of Patients")
plt.ylabel("Transition")

# Annotate with count + percentage
for i, (count, label) in enumerate(
    zip(transition_counts.values, transition_counts.index)
):
    percentage = 100 * count / total_transitions
    ax.text(
        count + 0.5,  # position slightly to the right of bar
        i,
        f"{count} ({percentage:.1f}%)",
        va="center",
    )

plt.tight_layout()
plt.show()


# print("Transition counts:")
# print(transition_counts)

### Retention

In [ ]:
# Ensure datetime
visite_date_column = "visit_date"
df[visite_date_column] = pd.to_datetime(df[visite_date_column], errors="coerce")

# Keep unique patient-event pairs
df_unique = df.drop_duplicates(subset=["PATNO", "EVENT_ID"]).copy()

# 1️⃣ Baseline date per patient
baseline_dates = (
    df_unique[df_unique["EVENT_ID"] == "BL"].groupby("PATNO")[visite_date_column].min()
)
df_unique["baseline_date"] = df_unique["PATNO"].map(baseline_dates)

# 2️⃣ Time from baseline in months
df_unique["months_from_baseline"] = (
    df_unique["visit_date"] - df_unique["baseline_date"]
).dt.days / 30.44
df_unique = df_unique[df_unique["months_from_baseline"] >= 0].dropna(
    subset=["months_from_baseline"]
)

# 3️⃣ Plot as a curve
df_plot = df_unique[df_unique["months_from_baseline"] <= 60].copy()
bins = np.arange(0, df_plot["months_from_baseline"].max() + 1, 3)
counts, edges = np.histogram(df_plot["months_from_baseline"], bins=bins)

# Midpoints of bins for plotting
bin_centers = (edges[:-1] + edges[1:]) / 2

plt.figure(figsize=(10, 6))
plt.plot(bin_centers, counts, marker="o", linestyle="-", color="steelblue")
plt.title(
    "Follow-up Visits Over Time (Months from Baseline) - bins of 3 months - up to 60 months"
)
plt.xlabel("Months from Baseline")
plt.ylabel("Number of Visits")
plt.grid(axis="y", alpha=0.3)
plt.xticks(np.arange(0, 60 + 1, 3))

plt.tight_layout()
plt.show()


In [ ]:
# Ensure datetime
df_unique = df.drop_duplicates(subset=["PATNO", "EVENT_ID"]).copy()
df_unique["visit_date"] = pd.to_datetime(df_unique["visit_date"], errors="coerce")

# Get baseline date per patient
baseline_dates = (
    df_unique[df_unique["EVENT_ID"] == "BL"].groupby("PATNO")["visit_date"].min()
)
df_unique["baseline_date"] = df_unique["PATNO"].map(baseline_dates)

# Compute months from baseline
df_unique["months_from_baseline"] = (
    df_unique["visit_date"] - df_unique["baseline_date"]
).dt.days / 30.44
df_unique = df_unique[df_unique["months_from_baseline"] >= 0]

# Compute total visits per patient
visits_per_patient = df_unique.groupby("PATNO")["EVENT_ID"].nunique()


# Assign subgroups based on total visits
def assign_visit_group(n):
    if n == 1:
        return "1 visit"
    elif n == 2:
        return "2 visits"
    else:
        return "3+ visits"
    # elif n == 3:
    #     return "3 visits"
    # elif n == 4:
    #     return "4 visits"

    # else:
    #     return "5+ visits"


df_unique["visit_group"] = (
    df_unique["PATNO"].map(visits_per_patient).map(assign_visit_group)
)
df_plot = df_unique[df_unique["months_from_baseline"] <= 60].copy()


# Plot
plt.figure(figsize=(10, 6))
bins = np.arange(0, df_plot["months_from_baseline"].max() + 1, 2)
for group_name, group_df in df_unique.groupby("visit_group"):
    counts, bin_edges = np.histogram(group_df["months_from_baseline"], bins=bins)
    # Convert counts to cumulative or normalized fraction if desired
    plt.plot(bin_edges[:-1], counts, marker="o", label=group_name)


# Plot full population
counts_all, _ = np.histogram(df_unique["months_from_baseline"], bins=bins)
plt.plot(
    bin_edges[:-1],
    counts_all,
    marker="o",
    color="black",
    linestyle="--",
    label="All patients",
)


plt.title("Follow-up Visits over Time by Patient Subgroup")
plt.xlabel("Months from Baseline")
plt.ylabel("Number of Visits")
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Patient Subgroup")
plt.xticks(np.arange(0, 60 + 1, 3))

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
bins = np.arange(0, df_plot["months_from_baseline"].max() + 1, 2)
# group_name = "visit_group"
group_name = "Research Group"
# Plot one curve per subgroup
for group_name, group_df in df_plot.groupby(group_name):
    # Get subgroup size (number of unique patients)
    n_patients = group_df["PATNO"].nunique()

    # Histogram counts
    counts, bin_edges = np.histogram(group_df["months_from_baseline"], bins=bins)

    # Convert to percentage relative to subgroup
    perc = counts / n_patients * 100

    plt.plot(bin_edges[:-1], perc, marker="o", label=group_name)

# Plot full population
total_patients = df_unique["PATNO"].nunique()
counts_all, _ = np.histogram(df_unique["months_from_baseline"], bins=bins)
perc_all = counts_all / total_patients * 100
plt.plot(
    bin_edges[:-1],
    perc_all,
    marker="o",
    color="black",
    linestyle="--",
    label="All patients",
)


plt.title("Follow-up Visits over Time by Patient Subgroup (≤ 60 months)")
plt.xlabel("Months from Baseline")
plt.ylabel("Percentage of Subgroup (%)")
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Patient Subgroup")

# X-axis ticks every 3 months
plt.xticks(np.arange(0, 60 + 1, 3))

plt.tight_layout()
plt.show()


- Tous on une baseline
- si 2 visits, la deuxieme est surtout à +12 et parfois +24 ou +48
- si 3, 2 suivant surtout sur +12 +24
- si 4, les trois apres BL sont equitablement réparties sur +12 +24 +48
- si 5+, bonne répartition avec du +36

# Image analysis

### Basic info on rows level

In [ ]:
df_image = df[df.columns.intersection(ida_df.columns)]
print("Image-related DataFrame shape:", df_image.shape)
print("Image-related DataFrame columns:", df_image.columns.tolist())

### histograms

In [ ]:
# Separate numeric and categorical columns
# numeric_cols = df_image.select_dtypes(include=np.number).columns
numeric_cols = ["Slice Thickness", "Matrix Z", "Field Strength"]
# numeric_cols = []

# -------------------------------
# Numeric Columns Histograms
# -------------------------------
for col in numeric_cols:
    plt.figure(figsize=(6, 4))

    # Plot histogram
    ax = sns.histplot(
        df_image[col].dropna(), bins=30, kde=False
    )  # disable KDE for counts clarity

    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")

    # Annotate counts on top of each bin
    for patch in ax.patches:
        height = patch.get_height()
        if height > 0:  # only annotate non-empty bins
            ax.text(
                patch.get_x() + patch.get_width() / 2,  # center of bin
                height + 0.5,  # slightly above the bar
                int(height),  # show integer count
                ha="center",
                va="bottom",
                fontsize=8,
            )

    plt.show()

# -------------------------------
# Categorical Columns Bar Plots
# -------------------------------
categorical_cols = [
    # "Visit",
    # "Sex",
    # "Research Group",
    "Modality",
    "Advanced_Modality",
    "Type",
    # "Structure",
    # "Laterality",
    # "Image Type",
    # "Registration",
    "Description",
    # "Tissue",
    # "Acquisition Plane",
    "Acquisition Type",
    # "Manufacturer",
    # "Mfg Model",
    # "Weighting",
]

categorical_cols = [col for col in categorical_cols if col in df_image.columns]

for col in categorical_cols:
    plt.figure(figsize=(6, 4))

    df_image[col] = df_image[col].fillna("Unknown").astype(str)

    # Compute counts after cleaning
    counts = df_image[col].value_counts()
    total = counts.sum()
    order = counts.index.tolist()  # already strings

    ax = sns.countplot(y=col, data=df_image, order=order)

    for p, category in zip(ax.patches, order):
        count = counts.get(category, 0)
        percent = 100 * count / total

        ax.text(
            p.get_width() + 0.5,
            p.get_y() + p.get_height() / 2,
            f"{count} ({percent:.1f}%)",
            va="center",
            fontsize=8,
        )

    plt.title(f"Value Counts for {col}")
    plt.xlabel("Count")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


### Image descriptions

In [ ]:
# COunt the values of "Image Description"
print(df_image["Description"].value_counts(dropna=False).head(10))

### Simultaneous Multimodal Availability (Per Visit)

In [ ]:
# Unique subject-visit-modality combinations
visit_modalities = (
    df.groupby(["PATNO", "EVENT_ID"])["Advanced_Modality"].unique().reset_index()
)

visit_modalities["n_modalities"] = visit_modalities["Advanced_Modality"].apply(len)
print(visit_modalities["n_modalities"].value_counts().to_frame().T)
print(visit_modalities["n_modalities"].describe().to_frame().T)


In [ ]:
visit_modalities.head(2)

In [ ]:
# Unique subject-visit-modality combinations
visit_modalities = (
    df.groupby(["PATNO", "EVENT_ID"])["Advanced_Modality"].unique().reset_index()
)

# Explode so each row is one modality per visit
exploded = visit_modalities.explode("Advanced_Modality")

# Count visits per modality
modality_counts = (
    exploded["Advanced_Modality"].value_counts().sort_values(ascending=False)
)

# Plot
plt.figure(figsize=(12, 6))
plt.bar(modality_counts.index, modality_counts.values)

plt.title("Number of Visits per Modality")
plt.xlabel("Advanced Modality")
plt.ylabel("Number of Visits")
plt.xticks(rotation=45, ha="right")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Create modality combination string
visit_modalities["modality_combo"] = visit_modalities["Advanced_Modality"].apply(
    lambda x: " | ".join(sorted(x))
)

# Count number of visits per combination
visit_counts = visit_modalities["modality_combo"].value_counts()

# Count number of unique patients per combination
patient_counts = (
    visit_modalities.groupby("modality_combo")["PATNO"]
    .nunique()
    .sort_values(ascending=False)
)

# Combine both in one table
summary = visit_counts.to_frame("n_visits").join(patient_counts.to_frame("n_patients"))

# print(summary)

In [ ]:
# Optional: keep only top 15 combinations for readability
top_n = 8
summary_plot = summary.sort_values("n_visits", ascending=False).head(top_n)

plt.figure(figsize=(10, 8))

ax = sns.barplot(x=summary_plot["n_visits"], y=summary_plot.index, color="steelblue")

plt.title("Top Modality Combinations per Visit")
plt.xlabel("Number of Visits")
plt.ylabel("Modality Combination")

# Annotate counts
for i, (visits, patients) in enumerate(
    zip(summary_plot["n_visits"], summary_plot["n_patients"])
):
    ax.text(visits + 1, i, f"{visits} visits | {patients} patients", va="center")

plt.tight_layout()
plt.show()

In [ ]:
# Pivot to binary presence matrix
visit_modality_matrix = (
    df.assign(present=1)
    .drop_duplicates(["PATNO", "EVENT_ID", "Advanced_Modality"])
    .pivot_table(
        index=["PATNO", "EVENT_ID"],
        columns="Advanced_Modality",
        values="present",
        fill_value=0,
    )
)
co_occurrence = visit_modality_matrix.T @ visit_modality_matrix
print(co_occurrence)

In [ ]:
df.head(2)

# Subject and Image IDs
Should be useful to download the data

In [ ]:
image_ids = (
    df["Image ID"]
    .dropna()  # remove NaNs
    .apply(
        lambda x: str(int(x)) if isinstance(x, float) else str(x).strip()
    )  # float → int → str, else str
    .unique()
    .tolist()
)

print(f"Unique Image IDs: {len(image_ids)}")
print(",".join(image_ids[: len(image_ids) // 2]))  # print first half
print(",".join(image_ids[len(image_ids) // 2 :]))  # print second half


In [ ]:
n = 5  # number of chunks you want

image_ids = (
    df["Image ID"]
    .dropna()
    .apply(lambda x: str(int(x)) if isinstance(x, float) else str(x).strip())
    .unique()
    .tolist()
)

print(f"Total unique Image IDs: {len(image_ids)}")

# Compute chunk size (ceiling division)
chunk_size = (len(image_ids) + n - 1) // n

for i in range(n):
    chunk = image_ids[i * chunk_size : (i + 1) * chunk_size]
    if not chunk:
        continue  # skip empty chunks if n > len(image_ids)

    print(f"\nChunk {i + 1}:")
    print(f"Length: {len(chunk)}")
    print(",".join(chunk))

In [ ]:
# Get unique patient IDs
patient_ids = (
    df["PATNO"]
    .dropna()  # remove NaNs
    .apply(lambda x: str(x).strip())  # convert to string and strip whitespace
    .unique()
    .tolist()
)

print(f"Unique patient IDs: {len(patient_ids)}")
print(",".join(patient_ids))